In [30]:
import math

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports · lumapi setup · Logging · I/O paths                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import sys, os, platform, time, logging
from pathlib import Path
from datetime import datetime
from typing import Optional, Tuple, Dict, Any

import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

LUMERICAL_VERSION = "v202"
if platform.system() == "Windows":
    LUMERICAL_ROOT = rf"C:\Program Files\Lumerical\{LUMERICAL_VERSION}"
    LUMERICAL_API  = rf"{LUMERICAL_ROOT}\api\python"
    LUMERICAL_BIN  = rf"{LUMERICAL_ROOT}\bin"
else:
    LUMERICAL_ROOT = f"/opt/lumerical/{LUMERICAL_VERSION}"
    LUMERICAL_API  = f"{LUMERICAL_ROOT}/api/python"
    LUMERICAL_BIN  = f"{LUMERICAL_ROOT}/bin"

if "lumapi" in sys.modules:
    del sys.modules["lumapi"]
if LUMERICAL_API not in sys.path:
    sys.path.insert(0, LUMERICAL_API)
if platform.system() == "Windows":
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(LUMERICAL_BIN))
    else:
        os.environ["PATH"] = str(LUMERICAL_BIN) + ";" + os.environ.get("PATH", "")

assert Path(LUMERICAL_API).exists(), f"API path not found: {LUMERICAL_API}"
assert Path(LUMERICAL_BIN).exists(), f"bin path not found: {LUMERICAL_BIN}"

import lumapi
print(f"lumapi imported from:\n  {lumapi.__file__}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s │ %(levelname)s │ %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("ICNT_Cascade")

VERSION_NAME = "ICNT_14Ring_Cascade_UniDir_neff_sweep_V2"
PROJECT_DIR  = Path.cwd()
DATA_DIR     = PROJECT_DIR / "data_ICNT_cascade_ring_sweep"
DATA_DIR.mkdir(parents=True, exist_ok=True)
HDF5_PATH    = DATA_DIR / f"{VERSION_NAME}.h5"
FIGURES_DIR  = DATA_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n  Data directory : {DATA_DIR}")
print(f"  HDF5 output    : {HDF5_PATH}")
print(f"  Figures dir    : {FIGURES_DIR}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 1
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Simulation parameters                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

N_RINGS = 14

RING_RADIUS_M = np.array([
    19.0021e-6, 19.1818e-6, 19.1934e-6, 19.2051e-6,
    19.2168e-6, 19.2284e-6, 19.2401e-6, 19.2517e-6,
    19.4158e-6, 19.4275e-6, 19.4393e-6, 19.4511e-6,
    19.4628e-6, 19.4746e-6,
])
RING_LAMBDA_RES_M = np.array([
    1.5500000e-6, 1.5500000e-6, 1.5507692e-6, 1.5515385e-6,
    1.5523077e-6, 1.5530769e-6, 1.5538462e-6, 1.5546154e-6,
    1.5553846e-6, 1.5561538e-6, 1.5569231e-6, 1.5576923e-6,
    1.5584615e-6, 1.5582308e-6,
])
RING_NEFF_TE = np.array([
    1.609803, 1.633303, 1.633121, 1.632939,
    1.632758, 1.632576, 1.632394, 1.632213,
    1.631974, 1.631792, 1.631611, 1.631430,
    1.631248, 1.631067,
])
RING_NG_TE = np.array([
    2.020543, 1.991101, 1.990956, 1.990808,
    1.990659, 1.990509, 1.990356, 1.990203,
    1.990047, 1.989891, 1.989733, 1.989691,
    1.989528, 1.989364,
])
RING_D_TE_PS2_PER_KM = np.zeros(N_RINGS)
RING_D_TM_PS2_PER_KM = np.zeros(N_RINGS)
RING_NEFF_TM         = np.full(N_RINGS, 1.7000)
RING_NG_TM           = np.full(N_RINGS, 2.1000)
RING_KAPPA_INPUT_SQ  = np.array([
    0.145778, 0.145072, 0.145011, 0.144949,
    0.144888, 0.144827, 0.144765, 0.144704,
    0.145696, 0.145634, 0.145572, 0.145518,
    0.145456, 0.145394,
])
RING_KAPPA_DROP_SQ = np.array([
    0.143402, 0.142672, 0.142609, 0.142546,
    0.142484, 0.142420, 0.142357, 0.142294,
    0.143269, 0.143205, 0.143142, 0.143086,
    0.143022, 0.142958,
])
RING_LOSS_DB_PER_M = np.full(N_RINGS, 101.0)
RING_POLARIZATION  = ["TE"] * N_RINGS

ONA_LAMBDA_START_M = 1.540e-6
ONA_LAMBDA_STOP_M  = 1.560e-6
ONA_N_POINTS       = 1000
ONA_POWER_DBM      = 0.0
ONA_N_INPUT_PORTS  = 2          # ONA monitors only RING_1 through and RING_14 through

SWEEP_N_POINTS = 21
SWEEP_NEFF = np.linspace(1.90, 2.10, SWEEP_N_POINTS)
SWEEP_NG   = np.linspace(2.20, 2.45, SWEEP_N_POINTS)

# ── Validation ────────────────────────────────────────────────────────────────
for arr, name in [
    (RING_RADIUS_M,      "RING_RADIUS_M"),
    (RING_LAMBDA_RES_M,  "RING_LAMBDA_RES_M"),
    (RING_NEFF_TE,       "RING_NEFF_TE"),
    (RING_NG_TE,         "RING_NG_TE"),
    (RING_KAPPA_INPUT_SQ,"RING_KAPPA_INPUT_SQ"),
    (RING_KAPPA_DROP_SQ, "RING_KAPPA_DROP_SQ"),
    (RING_LOSS_DB_PER_M, "RING_LOSS_DB_PER_M"),
]:
    assert len(arr) == N_RINGS, f"{name} length mismatch"
assert len(SWEEP_NEFF) == SWEEP_N_POINTS
assert len(SWEEP_NG)   == SWEEP_N_POINTS

print("=" * 72)
print("  INTERCONNECT 14-Ring Cascade — Parameter Summary  [UNIDIRECTIONAL]")
print("=" * 72)
print(f"  {'Ring':>5}  {'R [µm]':>9}  {'λ_res [nm]':>12}  "
      f"{'neff_TE':>9}  {'ng_TE':>8}  {'κ²_in':>8}  {'κ²_dr':>8}  {'Loss':>7}")
print("  " + "─" * 68)
for i in range(N_RINGS):
    tag = "  ← swept" if i == 0 else ""
    print(f"  {i+1:>5}  {RING_RADIUS_M[i]*1e6:>9.4f}  "
          f"{RING_LAMBDA_RES_M[i]*1e9:>12.4f}  "
          f"{RING_NEFF_TE[i]:>9.6f}  {RING_NG_TE[i]:>8.6f}  "
          f"{RING_KAPPA_INPUT_SQ[i]:>8.6f}  {RING_KAPPA_DROP_SQ[i]:>8.6f}  "
          f"{RING_LOSS_DB_PER_M[i]:>7.1f}{tag}")
print()
print(f"  ONA   : λ {ONA_LAMBDA_START_M*1e9:.2f}–{ONA_LAMBDA_STOP_M*1e9:.2f} nm  "
      f"│  {ONA_N_POINTS} pts  │  {ONA_POWER_DBM} dBm")
print(f"  Sweep : neff {SWEEP_NEFF[0]:.5f}→{SWEEP_NEFF[-1]:.5f}  "
      f"│  ng {SWEEP_NG[0]:.5f}→{SWEEP_NG[-1]:.5f}  │  {SWEEP_N_POINTS} pts")
print()
print("  CIRCUIT TOPOLOGY (CORRECTED)")
print("  ─────────────────────────────────────────────────────────────────")
print("  ONA output       → RING_1  input")
print("  RING_1  output 1 → ONA     input 1       (through → ONA)")
print("  RING_1  output 2 → RING_2  input          (drop   → cascade)")
print("  RING_n  output 1 → RING_n+1 input  n=2..13  (through → cascade)")
print("  RING_n  output 2 → OPM_n-1 input  n=2..14  (drop   → OPM)")
print("  RING_14 output 1 → ONA     input 2       (through → ONA)")
print("  RING_14 output 2 → OPM_13  input          (drop   → OPM)")
print("  ─────────────────────────────────────────────────────────────────")
print("  OPM_k monitors RING_(k+1) drop port,  k = 1..13")
print("=" * 72)

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 2
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  FIX — Replace _extract_results() in Cell 3 with this version             ║
# ║                                                                            ║
# ║  ROOT CAUSES (confirmed by debug output):                                  ║
# ║                                                                            ║
# ║  BUG 1 (blocker) — Wrong ONA result dict keys                             ║
# ║    INTERCONNECT v202 ONA dict keys confirmed:                              ║
# ║      'wavelength', 'frequency', 'TE transmission', 'Lumerical_dataset'    ║
# ║    OLD CODE used: raw_1["f"]   → KeyError: 'f'   ← this broke everything  ║
# ║    OLD CODE used: raw["T"]     → KeyError: 'T'                            ║
# ║    FIX: raw_1["frequency"]  and  raw["TE transmission"]                   ║
# ║                                                                            ║
# ║  BUG 2 — All 3 OPM result paths tried in debug failed                     ║
# ║    This version tries 10 candidate paths and caches the working one.      ║
# ║    It also queries INTERCONNECT directly for the available result names.   ║
# ║    If no path works, OPM data is NaN but ONA data is still fully valid.   ║
# ║                                                                            ║
# ║  IMPACT: Bug 1 caused KeyError in _extract_results() for EVERY sweep      ║
# ║  point → caught by the outer except → computed[i]=True but no data saved  ║
# ║  → wavelengths_m stayed None for the entire 21-point sweep.               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝


# ── Candidate OPM result paths — ordered by likelihood in INTERCONNECT v202 ──
_OPM_CANDIDATE_PATHS = [
    "input/mode 1/power",        # standard port-mode path (most common >= v221)
    "mode 1/power",              # without explicit port prefix
    "input 1/mode 1/power",      # numbered port variant
    "input/TE power",            # TE-labelled power
    "mode 1/TE power",
    "TE power",
    "power",                     # bare key (some older builds)
    "input/power",
    "input/mode 1/transmission", # some OPMs report as "transmission"
    "mode 1/transmission",
]

# Per-INTERCONNECT-session OPM path cache: {id(ic): working_path_or_empty}
_OPM_PATH_CACHE: dict = {}


def _discover_opm_path(ic, test_opm: str) -> str:
    """
    Find the first working getresult() path for an OPM element.

    Also queries INTERCONNECT's own result-list function so the
    working path appears in the log even if none of our candidates match.

    Returns the working path string, or '' if none found.
    """
    # ── Ask INTERCONNECT to report what result names exist ───────────────────
    for query in (
        f'?getresult("{test_opm}");',
        f'?listresult("{test_opm}");',
    ):
        try:
            avail = ic.eval(query)
            if avail and str(avail).strip():
                log.info(f"  {test_opm} INTERCONNECT-listed results: "
                         f"{str(avail).strip()!r}")
                break
        except Exception:
            pass

    # ── Try each candidate path ───────────────────────────────────────────────
    for path in _OPM_CANDIDATE_PATHS:
        try:
            raw = ic.getresult(test_opm, path)
            if raw is not None:
                log.info(f"  OPM result path discovered and cached: '{path}'")
                return path
        except Exception:
            continue

    log.warning(
        f"  [{test_opm}] No working result path found in {len(_OPM_CANDIDATE_PATHS)} "
        f"candidates.  OPM spectra will be NaN for this sweep.\n"
        f"  To fix: run the following in a new cell after a successful run():\n"
        f'      ic.eval(\'?getresult("{test_opm}");\')  '
        f"and add the returned path to _OPM_CANDIDATE_PATHS."
    )
    return ""


def _extract_results(ic) -> tuple:
    """
    Extract ONA transmission + OPM spectra after a successful run().

    CONFIRMED INTERCONNECT v202 result structure
    ─────────────────────────────────────────────
    ONA getresult("ONA_1", "input N/mode 1/transmission") returns a dict:
      {
        "wavelength"       : ndarray (n_wl,)   [m]
        "frequency"        : ndarray (n_wl,)   [Hz]   ← was wrongly accessed as "f"
        "TE transmission"  : ndarray (n_wl,)   [linear, 0–1]  ← was wrongly as "T"
        "Lumerical_dataset": metadata
      }

    Returns
    -------
    wl_m          : (n_wl,)          wavelengths [m], ascending
    T_port1_dB    : (n_wl,)          ONA input 1 — RING_1  through [dB]
    T_port2_dB    : (n_wl,)          ONA input 2 — RING_14 through [dB]
    opm_power_dBm : (N_OPM,)         mean power per OPM [dBm]
    opm_spectrum_W: (N_OPM, n_wl)    spectral power per OPM [W]

    OPM data is NaN when no valid result path can be found (ONA data unaffected).
    """
    # ── ONA: frequency axis (CONFIRMED key: "frequency", not "f") ────────────
    raw_1  = ic.getresult(ONA_NAME, "input 1/mode 1/transmission")
    f_arr  = np.asarray(raw_1["frequency"]).flatten()   # ← FIX: was raw_1["f"]
    sort_i = np.argsort(f_arr)[::-1]                    # high→low f = ascending λ
    wl_m   = SPEED_OF_LIGHT / f_arr[sort_i]
    n_wl   = len(wl_m)

    # ── ONA transmissions (CONFIRMED key: "TE transmission", not "T") ─────────
    def _T_dB(port_label: str) -> np.ndarray:
        raw = ic.getresult(ONA_NAME, f"{port_label}/mode 1/transmission")
        T   = np.asarray(raw["TE transmission"]).flatten()[sort_i]  # ← FIX: was raw["T"]
        T   = np.where(np.abs(T) > 0, np.abs(T), 1e-30)
        return 10.0 * np.log10(T)

    T_port1_dB = _T_dB("input 1")
    T_port2_dB = _T_dB("input 2")

    # ── OPM spectra (13 meters) ───────────────────────────────────────────────
    opm_power_dBm  = np.full(N_OPM, np.nan)
    opm_spectrum_W = np.full((N_OPM, n_wl), np.nan)

    # Discover working OPM path once per INTERCONNECT session
    ic_id = id(ic)
    if ic_id not in _OPM_PATH_CACHE:
        _OPM_PATH_CACHE[ic_id] = _discover_opm_path(ic, opm_name(1))
    opm_path = _OPM_PATH_CACHE[ic_id]

    if not opm_path:
        # ONA data is valid — return it with NaN OPM fields
        return wl_m, T_port1_dB, T_port2_dB, opm_power_dBm, opm_spectrum_W

    # ── Extract each OPM ──────────────────────────────────────────────────────
    for k in range(1, N_OPM + 1):
        on  = opm_name(k)
        idx = k - 1
        try:
            raw_opm = ic.getresult(on, opm_path)

            # Handle both dict-form and raw array-form results
            if isinstance(raw_opm, dict):
                p_vals = None
                for pkey in ("power", "TE power", "TE transmission",
                             "power_abs", "P", "Pt"):
                    if pkey in raw_opm:
                        p_vals = np.asarray(raw_opm[pkey]).flatten()
                        break
                if p_vals is None:
                    # Fallback: use the last non-metadata value in the dict
                    candidates = [
                        v for dk, v in raw_opm.items()
                        if dk not in ("wavelength", "frequency", "Lumerical_dataset")
                    ]
                    if candidates:
                        p_vals = np.asarray(candidates[-1]).flatten()
            else:
                p_vals = np.asarray(raw_opm).flatten()

            if p_vals is None or p_vals.size == 0:
                continue

            # Assign to spectrum array (handle any size match)
            if p_vals.size == n_wl:
                opm_spectrum_W[idx, :] = p_vals[sort_i]
            elif p_vals.size == f_arr.size:
                opm_spectrum_W[idx, :] = p_vals[sort_i]
            elif p_vals.size == 1:
                opm_spectrum_W[idx, :] = float(p_vals[0])
            else:
                opm_spectrum_W[idx, :] = float(p_vals.mean())

            mean_p = float(np.nanmean(opm_spectrum_W[idx, :]))
            opm_power_dBm[idx] = 10.0 * np.log10(max(mean_p, 1e-40) * 1e3)

        except Exception as exc:
            log.warning(f"  {on} extraction failed at path '{opm_path}': {exc}")

    return wl_m, T_port1_dB, T_port2_dB, opm_power_dBm, opm_spectrum_W


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  OPM PATH FINDER — Run this cell ONCE after a successful run()             ║
# ║  to interactively find the correct OPM result path for your INTERCONNECT   ║
# ║  version.  Paste the result into _OPM_CANDIDATE_PATHS[0].                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def probe_opm_paths_interactive(hide_gui: bool = False) -> None:
    """
    Launch INTERCONNECT, build the circuit, run one point, then exhaustively
    probe every candidate OPM result path and print the dict keys of any
    that succeed.  Also queries INTERCONNECT's own result lister.

    Run this once when OPM paths are unknown for a new INTERCONNECT version.
    """
    print("Launching INTERCONNECT for OPM path probe …")
    ic = lumapi.INTERCONNECT(hide=hide_gui)
    try:
        _build_circuit(ic)
        _eval(ic, "switchtodesign")
        _update_ring1_neff_ng(ic, float(SWEEP_NEFF[0]), float(SWEEP_NG[0]))
        _eval(ic, "run")
        print("run() OK.  Probing OPM_1 result paths …\n")

        # INTERCONNECT-side query
        for q in ('?getresult("OPM_1");', '?listresult("OPM_1");'):
            try:
                r = ic.eval(q)
                print(f"  {q!r} → {r!r}")
            except Exception as e:
                print(f"  {q!r} → FAIL: {e}")

        print()
        extended_paths = _OPM_CANDIDATE_PATHS + [
            "input/mode 1/power_abs",
            "input/mode 1/Pt",
            "input/mode 1/P",
            "port 1/mode 1/power",
            "port 1/mode 1/transmission",
        ]
        print(f"  {'PATH':<35}  RESULT")
        print("  " + "─" * 55)
        for path in extended_paths:
            try:
                raw = ic.getresult("OPM_1", path)
                if isinstance(raw, dict):
                    info = f"dict  keys={list(raw.keys())}"
                else:
                    arr  = np.asarray(raw).flatten()
                    info = f"array size={arr.size}"
                print(f"  {path:<35}  ✓  {info}")
            except Exception as e:
                short_e = str(e)[:60]
                print(f"  {path:<35}  ✗  {short_e}")
    finally:
        try:
            ic.close()
        except Exception:
            pass
    print("\nDone.  Add the ✓ path to _OPM_CANDIDATE_PATHS[0] in your code.")


# Uncomment to run:
# probe_opm_paths_interactive(hide_gui=False)

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Post-processing · Visualisation                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

plt.rcParams.update({
    "font.family"    : "serif",
    "font.serif"     : ["DejaVu Serif", "Georgia", "Times New Roman"],
    "font.size"      : 11, "axes.labelsize": 12, "axes.titlesize": 13,
    "legend.fontsize": 9,  "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.linewidth" : 0.8, "axes.grid": True, "grid.alpha": 0.3,
    "grid.linewidth" : 0.5, "lines.linewidth": 1.4,
    "figure.dpi"     : 120, "savefig.dpi": 300, "savefig.bbox": "tight",
})


# ─────────────────────────────────────────────────────────────────────────────
# Data loader helpers
# ─────────────────────────────────────────────────────────────────────────────
def load_results(path: Path = HDF5_PATH) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None
    try:
        with h5py.File(path, "r") as f:
            if "metadata/wavelengths_m" not in f:
                return None
            wl = f["metadata/wavelengths_m"][:]
            if wl is None or len(wl) == 0:
                return None
            return dict(
                neff_sweep     = f["metadata/neff_sweep"][:],
                ng_sweep       = f["metadata/ng_sweep"][:],
                wavelengths_m  = wl,
                T_port1_dB     = f["results/T_port1_dB"][:],
                T_port2_dB     = f["results/T_port2_dB"][:],
                opm_power_dBm  = f["results/opm_power_dBm"][:],
                opm_spectrum_W = f["results/opm_spectrum_W"][:],
                computed       = f["flags/computed"][:],
            )
    except Exception as exc:
        log.warning(f"Could not read HDF5 ({exc}): {path}")
        return None


def get_results(path: Path = HDF5_PATH) -> Dict[str, Any]:
    mem_reason = ""
    try:
        r  = sweep_results
        wl = r.get("wavelengths_m")
        if wl is not None and len(wl) > 0:
            return r
        mem_reason = "sweep_results in memory but wavelengths_m is None"
    except NameError:
        mem_reason = "sweep_results not defined (Cell 3 not run)"
    r_hdf5 = load_results(path)
    if r_hdf5 is not None:
        log.info(f"Loaded from HDF5: {int(r_hdf5['computed'].sum())}/{SWEEP_N_POINTS} pts.")
        return r_hdf5
    hdf5_reason = (
        f"HDF5 exists but empty — delete and re-run Cell 3:\n    {path}"
        if path.exists() else f"HDF5 not found:\n    {path}"
    )
    raise RuntimeError(
        f"\n{'='*65}\n  No results available.\n"
        f"  Memory : {mem_reason}\n  HDF5   : {hdf5_reason}\n"
        f"  ► Run Cell 3.\n{'='*65}"
    )


def _valid_mask(r: Dict) -> np.ndarray:
    return r["computed"].astype(bool)


# ─────────────────────────────────────────────────────────────────────────────
# Plot 1 — ONA transmission spectra (port 1 or 2), colour = neff_1
# ─────────────────────────────────────────────────────────────────────────────
def plot_transmittance_sweep(
    results=None, port: int = 1, n_curves: int = 8,
    figsize=(10, 5), cmap_name: str = "plasma", save: bool = True,
) -> plt.Figure:
    """
    Plot selected transmission spectra from the ONA.
    port=1 → RING_1 through (ONA input 1)
    port=2 → RING_14 through (ONA input 2)
    """
    if results is None:
        results = get_results()
    neff_arr  = results["neff_sweep"]
    wl_nm     = results["wavelengths_m"] * 1e9
    T_data    = results["T_port1_dB"] if port == 1 else results["T_port2_dB"]
    mask      = _valid_mask(results)
    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[
        np.round(np.linspace(0, len(valid_idx) - 1, n_sel)).astype(int)
    ]
    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=neff_arr[sel_idx].min(), vmax=neff_arr[sel_idx].max())
    sm   = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(neff_arr[idx])), lw=1.2, alpha=0.88)

    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Ring 1 — $n_{eff}$ (TE)", fontsize=11)

    port_label = ("RING_1 through  →  ONA input 1" if port == 1
                  else "RING_14 through  →  ONA input 2")
    ax.set_xlabel("Wavelength  (nm)")
    ax.set_ylabel("Transmission  (dB)")
    ax.set_title(
        f"ONA Transmission — {port_label}  [Unidirectional]\n"
        f"({n_sel} of {int(mask.sum())} sweep pts,  colour = $n_{{eff,1}}$)"
    )
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        stem = f"transmittance_port{port}_{n_sel}curves"
        fig.savefig(FIGURES_DIR / f"{stem}.png")
        fig.savefig(FIGURES_DIR / f"{stem}.pdf")
        log.info(f"Saved → {stem}.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 2 — Mean drop power (dBm) vs neff for each OPM
# ─────────────────────────────────────────────────────────────────────────────
def plot_opm_power_vs_neff(
    results=None, figsize=(10, 5), save: bool = True,
) -> plt.Figure:
    """
    Plot mean power vs Ring-1 neff for all 13 OPMs in a single figure.
    Each line corresponds to one OPM (= one ring's drop port).
    """
    if results is None:
        results = get_results()
    neff_arr  = results["neff_sweep"]
    opm_power = results["opm_power_dBm"]   # shape (n_pts, 13)
    mask      = _valid_mask(results)
    neff_v    = neff_arr[mask]
    opm_v     = opm_power[mask, :]         # (n_valid, 13)

    cmap = plt.get_cmap("tab20")
    fig, ax = plt.subplots(figsize=figsize)

    for k in range(1, N_OPM + 1):
        ring_monitored = k + 1
        ax.plot(
            neff_v, opm_v[:, k - 1],
            color=cmap(k / N_OPM), lw=1.3,
            marker="o", ms=3, alpha=0.85,
            label=f"OPM_{k}  (RING_{ring_monitored} drop)",
        )

    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Mean power  (dBm)")
    ax.set_title(
        "Drop Power vs Ring 1 $n_{eff}$  [All 13 OPMs, Unidirectional]\n"
        "OPM_k monitors RING_{k+1} output 2 (drop)"
    )
    ax.legend(ncol=2, framealpha=0.85, fontsize=8, loc="best")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fig.savefig(FIGURES_DIR / "opm_power_vs_neff_all.png")
        fig.savefig(FIGURES_DIR / "opm_power_vs_neff_all.pdf")
        log.info("Saved → opm_power_vs_neff_all.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 3 — Drop spectrum heatmap for a single OPM (neff × wavelength)
# ─────────────────────────────────────────────────────────────────────────────
def plot_opm_spectrum_heatmap(
    results=None, opm_k: int = 13,
    figsize=(10, 3.5), cmap_name: str = "inferno",
    vmin_dBm=None, vmax_dBm=None, save: bool = True,
) -> plt.Figure:
    """
    Heatmap: neff (y) × wavelength (x) of power at OPM_opm_k [dBm].
    opm_k is 1-based (1..13).  Default = OPM_13 (RING_14 drop).
    """
    if results is None:
        results = get_results()
    assert 1 <= opm_k <= N_OPM, f"opm_k must be 1..{N_OPM}"
    ring_monitored = opm_k + 1

    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    opm_spec = results["opm_spectrum_W"]          # (n_pts, N_OPM, n_wl)
    mask     = _valid_mask(results)
    neff_v   = neff_arr[mask]
    spec_v   = opm_spec[mask, opm_k - 1, :]       # (n_valid, n_wl)
    spec_dBm = 10.0 * np.log10(np.where(spec_v > 0, spec_v, 1e-30) * 1e3)

    _vmin = vmin_dBm if vmin_dBm is not None else np.nanpercentile(spec_dBm, 2)
    _vmax = vmax_dBm if vmax_dBm is not None else np.nanpercentile(spec_dBm, 98)

    fig, ax = plt.subplots(figsize=figsize)
    img = ax.pcolormesh(wl_nm, neff_v, spec_dBm,
                        cmap=cmap_name, vmin=_vmin, vmax=_vmax, shading="auto")
    cbar = fig.colorbar(img, ax=ax, pad=0.01)
    cbar.set_label("Power  (dBm)", fontsize=10)
    ax.set_xlabel("Wavelength  (nm)")
    ax.set_ylabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_title(
        f"Drop Spectrum — OPM_{opm_k}  (RING_{ring_monitored} drop)  "
        f"[Unidirectional]\n"
        f"neff sweep on Ring 1"
    )
    fig.tight_layout()

    if save:
        fname = f"opm{opm_k}_spectrum_heatmap"
        fig.savefig(FIGURES_DIR / f"{fname}.png")
        fig.savefig(FIGURES_DIR / f"{fname}.pdf")
        log.info(f"Saved → {fname}.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 4 — All OPM heatmaps in one figure (grid)
# ─────────────────────────────────────────────────────────────────────────────
def plot_all_opm_heatmaps(
    results=None, ncols: int = 4,
    cmap_name: str = "inferno", save: bool = True,
) -> plt.Figure:
    """
    Grid of N_OPM=13 heatmaps, one per OPM.
    Useful to see how the drop spectral shape evolves along the cascade.
    """
    if results is None:
        results = get_results()
    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    opm_spec = results["opm_spectrum_W"]
    mask     = _valid_mask(results)
    neff_v   = neff_arr[mask]

    nrows = math.ceil(N_OPM / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 3.8, nrows * 2.8),
                             sharex=True, sharey=True)
    axes_flat = axes.flatten()

    # Global colour range
    all_dBm = []
    for k in range(N_OPM):
        sv = opm_spec[mask, k, :]
        sd = 10.0 * np.log10(np.where(sv > 0, sv, 1e-30) * 1e3)
        all_dBm.append(sd)
    all_dBm = np.concatenate([a.ravel() for a in all_dBm])
    vmin    = np.nanpercentile(all_dBm, 2)
    vmax    = np.nanpercentile(all_dBm, 98)

    for k in range(1, N_OPM + 1):
        ax   = axes_flat[k - 1]
        spec_dBm = all_dBm  # reuse per-OPM slice
        sv   = opm_spec[mask, k - 1, :]
        sd   = 10.0 * np.log10(np.where(sv > 0, sv, 1e-30) * 1e3)
        im   = ax.pcolormesh(wl_nm, neff_v, sd,
                             cmap=cmap_name, vmin=vmin, vmax=vmax, shading="auto")
        ax.set_title(f"OPM_{k}  (R{k+1} drop)", fontsize=8)
        if k % ncols == 1:
            ax.set_ylabel("$n_{eff,1}$", fontsize=7)
        if k > (nrows - 1) * ncols:
            ax.set_xlabel("λ (nm)", fontsize=7)
        ax.tick_params(labelsize=6)

    # Hide unused subplots
    for ax in axes_flat[N_OPM:]:
        ax.set_visible(False)

    fig.colorbar(im, ax=axes_flat[:N_OPM], shrink=0.6, pad=0.02,
                 label="Power (dBm)", fraction=0.015)
    fig.suptitle(
        "All OPM Drop Spectra  [14-Ring Unidirectional Cascade]\n"
        "neff sweep on Ring 1",
        fontsize=11,
    )
    fig.tight_layout()

    if save:
        fig.savefig(FIGURES_DIR / "all_opm_heatmaps_grid.png", dpi=200)
        fig.savefig(FIGURES_DIR / "all_opm_heatmaps_grid.pdf")
        log.info("Saved → all_opm_heatmaps_grid.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 5 — Resonance tracking (transmission dip wavelength vs neff)
# ─────────────────────────────────────────────────────────────────────────────
def plot_resonance_tracking(
    results=None, port: int = 1, figsize=(7, 4.5), save: bool = True,
) -> plt.Figure:
    if results is None:
        results = get_results()
    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    T_data   = results["T_port1_dB"] if port == 1 else results["T_port2_dB"]
    mask     = _valid_mask(results)
    neff_v   = neff_arr[mask]
    T_v      = T_data[mask, :]
    dip_idx  = np.argmin(T_v, axis=1)
    lam_dip  = wl_nm[dip_idx]
    coeffs   = np.polyfit(neff_v, lam_dip, 1)
    sens     = coeffs[0]

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(neff_v, lam_dip, s=20, zorder=5, color="#2166ac", label="Resonance dip")
    ax.plot(neff_v, np.poly1d(coeffs)(neff_v), "r--", lw=1.5,
            label=f"Linear fit   ∂λ/∂n = {sens:.3f} nm / RIU")
    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Resonance wavelength  (nm)")
    ax.set_title(
        f"Resonance Tracking — ONA Port {port}  [Unidirectional]\n"
        f"Sensitivity: {sens:.3f} nm / RIU"
    )
    ax.legend(framealpha=0.9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fname = f"resonance_tracking_port{port}"
        fig.savefig(FIGURES_DIR / f"{fname}.png")
        fig.savefig(FIGURES_DIR / f"{fname}.pdf")
        log.info(f"Saved → {fname}.png/pdf")
    log.info(f"Resonance sensitivity (port {port}): {sens:.4f} nm/RIU")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Run all plots
# ─────────────────────────────────────────────────────────────────────────────
try:
    _res = get_results()
except RuntimeError as exc:
    print(str(exc))
    raise
else:
    fig1 = plot_transmittance_sweep(_res, port=1, n_curves=8)
    fig2 = plot_transmittance_sweep(_res, port=2, n_curves=8)
    fig3 = plot_opm_power_vs_neff(_res)
    fig4 = plot_opm_spectrum_heatmap(_res, opm_k=13)   # RING_14 drop
    fig5 = plot_opm_spectrum_heatmap(_res, opm_k=1)    # RING_2  drop
    fig6 = plot_all_opm_heatmaps(_res)
    fig7 = plot_resonance_tracking(_res, port=1)
    plt.show()
    print(f"\n  Figures → {FIGURES_DIR}")
    print(f"  HDF5    → {HDF5_PATH}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 4
# ═════════════════════════════════════════════════════════════════════════════

lumapi imported from:
  C:\Program Files\Lumerical\v202\api\python\lumapi.py

  Data directory : C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep
  HDF5 output    : C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\ICNT_14Ring_Cascade_UniDir_neff_sweep_V2.h5
  Figures dir    : C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\figures
  INTERCONNECT 14-Ring Cascade — Parameter Summary  [UNIDIRECTIONAL]
   Ring     R [µm]    λ_res [nm]    neff_TE     ng_TE     κ²_in     κ²_dr     Loss
  ────────────────────────────────────────────────────────────────────
      1    19.0021     1550.0000   1.609803  2.020543  0.145778  0.143402    101.0  ← swept
      2    19.1818     1550.0000   1.633303  1.991101  0.145072  0.142672    101.0
      3    19.1934     1550.7692   1.633

RuntimeError: 
=================================================================
  No results available.
  Memory : sweep_results in memory but wavelengths_m is None
  HDF5   : HDF5 not found:
    C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\ICNT_14Ring_Cascade_UniDir_neff_sweep_V2.h5
  ► Run Cell 3.
=================================================================

In [25]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  DEBUG CELL — paste into a NEW notebook cell and run it                    ║
# ║  Requires: Cell 1, Cell 2, Cell 3 (at least up to function definitions)   ║
# ║  Purpose : find EXACTLY which of the 8 steps fails and WHY                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def run_single_point_debug(neff_val: float, ng_val: float,
                           hide_gui: bool = False) -> None:
    import traceback

    print("=" * 58)
    print(f"  SINGLE-POINT DEBUG")
    print(f"  neff = {neff_val:.6f}   ng = {ng_val:.6f}")
    print("=" * 58)

    # ── STEP 1: launch ────────────────────────────────────────────────────────
    print("\n[1] Launching INTERCONNECT …")
    try:
        ic = lumapi.INTERCONNECT(hide=hide_gui)
        print("    OK")
    except Exception:
        print("    FAILED — INTERCONNECT did not launch")
        traceback.print_exc()
        return

    try:
        # ── STEP 2: build circuit ─────────────────────────────────────────────
        print("\n[2] Building circuit (_build_circuit) …")
        try:
            _build_circuit(ic)
            print("    OK — no exception raised")
        except Exception:
            print("    FAILED — _build_circuit raised:")
            traceback.print_exc()
            return

        # ── STEP 3: probe element names ───────────────────────────────────────
        print("\n[3] Checking element names exist in schematic …")
        for elem in [ONA_NAME,
                     ring_name(1), ring_name(2), ring_name(N_RINGS),
                     opm_name(1),  opm_name(N_OPM)]:
            ok = _try_eval(ic, f'select("{elem}")')
            status = "✓ found" if ok else "✗ NOT FOUND"
            print(f"    {elem:<15}  {status}")

        # ── STEP 4: switchtodesign + param update ─────────────────────────────
        print("\n[4] switchtodesign + update Ring 1 neff/ng …")
        try:
            _eval(ic, "switchtodesign")
            _update_ring1_neff_ng(ic, neff_val, ng_val)
            print("    OK")
        except Exception:
            print("    FAILED:")
            traceback.print_exc()
            return

        # ── STEP 5: run ───────────────────────────────────────────────────────
        print("\n[5] ic.eval('run') …")
        try:
            _eval(ic, "run")
            print("    OK — simulation finished")
        except Exception as exc:
            print(f"\n    !! RUN FAILED — this is your blocking error:")
            print(f"    {exc}")
            # Try to retrieve the last INTERCONNECT-side error message
            for err_cmd in ("?lasterror;", "?getlasterror();"):
                try:
                    msg = ic.eval(err_cmd)
                    if msg:
                        print(f"    INTERCONNECT lasterror: {msg}")
                    break
                except Exception:
                    pass
            print()
            print("    Common causes for RUN FAILED in a cascade circuit:")
            print("    A) An OPM or ring port is left unconnected")
            print("    B) connect() silently failed for one wire")
            print("    C) INTERCONNECT licence / solver issue")
            print("    → Share the error text above with the next prompt.")
            return

        # ── STEP 6: probe ONA result paths ───────────────────────────────────
        print("\n[6] Probing ONA result paths …")
        ona_paths = [
            "input 1/mode 1/transmission",
            "input 2/mode 1/transmission",
            "input 1/mode 1/T_abs",
            "input 1",
        ]
        for path in ona_paths:
            try:
                r = ic.getresult(ONA_NAME, path)
                info = (f"dict keys={list(r.keys())}"
                        if isinstance(r, dict) else f"type={type(r).__name__}")
                print(f"    '{path}'  →  OK  ({info})")
            except Exception as e:
                print(f"    '{path}'  →  FAIL: {e}")

        # ── STEP 7: probe OPM result paths ────────────────────────────────────
        print("\n[7] Probing OPM_1 result paths …")
        opm_paths = ["input/mode 1/power", "power", "input"]
        for path in opm_paths:
            try:
                r = ic.getresult(opm_name(1), path)
                if isinstance(r, dict):
                    info = f"dict keys={list(r.keys())}"
                else:
                    arr = np.asarray(r).flatten()
                    info = f"array size={arr.size}  min={arr.min():.3e}"
                print(f"    '{path}'  →  OK  ({info})")
            except Exception as e:
                print(f"    '{path}'  →  FAIL: {e}")

        # ── STEP 8: full _extract_results ────────────────────────────────────
        print("\n[8] _extract_results() …")
        try:
            wl_m, t1, t2, opm_p, opm_s = _extract_results(ic)
            print(f"    OK")
            print(f"    wavelengths : {len(wl_m)} pts  "
                  f"{wl_m[0]*1e9:.3f}–{wl_m[-1]*1e9:.3f} nm")
            print(f"    T_port1_dB  : {t1.min():.2f} … {t1.max():.2f} dB")
            print(f"    T_port2_dB  : {t2.min():.2f} … {t2.max():.2f} dB")
            print(f"    opm_power_dBm (13 OPMs):")
            for k in range(N_OPM):
                print(f"      OPM_{k+1}  (RING_{k+2} drop) = {opm_p[k]:.2f} dBm")
        except Exception:
            print("    FAILED:")
            traceback.print_exc()

    finally:
        try:
            ic.close()
        except Exception:
            pass
        print("\n  INTERCONNECT session closed.")
        print("=" * 58)


# ── Run debug on the FIRST sweep point ────────────────────────────────────────
run_single_point_debug(
    neff_val = float(SWEEP_NEFF[0]),
    ng_val   = float(SWEEP_NG[0]),
    hide_gui = False,           # keep False so you can see the INTERCONNECT window
)

  SINGLE-POINT DEBUG
  neff = 1.900000   ng = 2.200000

[1] Launching INTERCONNECT …
    OK

[2] Building circuit (_build_circuit) …


17:35:22 │ INFO │   ONA_1 added and configured.
17:35:22 │ INFO │   RING_ 1 added  [unidir, neff=1.609803, L=119.3937 µm]
17:35:23 │ INFO │   RING_ 2 added  [unidir, neff=1.633303, L=120.5228 µm]
17:35:23 │ INFO │   RING_ 3 added  [unidir, neff=1.633121, L=120.5957 µm]
17:35:23 │ INFO │   RING_ 4 added  [unidir, neff=1.632939, L=120.6692 µm]
17:35:23 │ INFO │   RING_ 5 added  [unidir, neff=1.632758, L=120.7427 µm]
17:35:23 │ INFO │   RING_ 6 added  [unidir, neff=1.632576, L=120.8156 µm]
17:35:24 │ INFO │   RING_ 7 added  [unidir, neff=1.632394, L=120.8891 µm]
17:35:24 │ INFO │   RING_ 8 added  [unidir, neff=1.632213, L=120.9620 µm]
17:35:24 │ INFO │   RING_ 9 added  [unidir, neff=1.631974, L=121.9931 µm]
17:35:24 │ INFO │   RING_10 added  [unidir, neff=1.631792, L=122.0666 µm]
17:35:24 │ INFO │   RING_11 added  [unidir, neff=1.631611, L=122.1407 µm]
17:35:24 │ INFO │   RING_12 added  [unidir, neff=1.631430, L=122.2149 µm]
17:35:25 │ INFO │   RING_13 added  [unidir, neff=1.631248, L=122

    OK — no exception raised

[3] Checking element names exist in schematic …
    ONA_1            ✓ found
    RING_1           ✓ found
    RING_2           ✓ found
    RING_14          ✓ found
    OPM_1            ✓ found
    OPM_13           ✓ found

[4] switchtodesign + update Ring 1 neff/ng …
    OK

[5] ic.eval('run') …
    OK — simulation finished

[6] Probing ONA result paths …
    'input 1/mode 1/transmission'  →  OK  (dict keys=['wavelength', 'frequency', 'TE transmission', 'Lumerical_dataset'])
    'input 2/mode 1/transmission'  →  OK  (dict keys=['wavelength', 'frequency', 'TE transmission', 'Lumerical_dataset'])
    'input 1/mode 1/T_abs'  →  FAIL: "Can not find result 'input 1/mode 1/T_abs' in the result provider 'ONA_1'"
    'input 1'  →  FAIL: "Can not find result 'input 1' in the result provider 'ONA_1'"

[7] Probing OPM_1 result paths …
    'input/mode 1/power'  →  FAIL: "Can not find result 'input/mode 1/power' in the result provider 'OPM_1'"
    'power'  →  FAIL: "Ca

Traceback (most recent call last):
  File "C:\Users\jero0\AppData\Local\Temp\ipykernel_13864\4249593936.py", line 117, in run_single_point_debug
    wl_m, t1, t2, opm_p, opm_s = _extract_results(ic)
                                 ^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jero0\AppData\Local\Temp\ipykernel_13864\616606131.py", line 132, in _extract_results
    f_arr  = np.asarray(raw_1["f"]).flatten()
                        ~~~~~^^^^^
KeyError: 'f'



  INTERCONNECT session closed.


In [14]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  DIAGNÓSTICO — Port names + connect() syntax                               ║
# ║  Tests: ONA ↔ RING ↔ OPM with all candidate port name combinations        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

ic_probe = lumapi.INTERCONNECT(hide=False)

SPEED_OF_LIGHT = 299_792_458.0

def _safe_eval(ic, cmd, label=""):
    cmd = cmd.strip().rstrip(";") + ";"
    try:
        ic.eval(cmd)
        print(f"  ✓  {label or cmd}")
        return True
    except Exception as e:
        print(f"  ✗  {label or cmd}  →  {e}")
        return False

try:
    # ─────────────────────────────────────────────────────────────────────────
    # 0. Fresh canvas
    # ─────────────────────────────────────────────────────────────────────────
    ic_probe.eval("switchtodesign; selectall; delete;")
    print("✓ Canvas cleared\n")

    # ─────────────────────────────────────────────────────────────────────────
    # 1. Add elements
    # ─────────────────────────────────────────────────────────────────────────
    print("── Add elements ─────────────────────────────────────────────────")
    _safe_eval(ic_probe, 'addelement("Optical Network Analyzer"); set("name","ONA_1");',
               "Add ONA_1")
    _safe_eval(ic_probe, 'addelement("Double Bus Ring Resonator"); set("name","RING_1");',
               "Add RING_1")
    _safe_eval(ic_probe, 'addelement("Optical Power Meter"); set("name","OPM_1");',
               "Add OPM_1")

    # ─────────────────────────────────────────────────────────────────────────
    # 2. Read all port-related properties via getnamed
    # ─────────────────────────────────────────────────────────────────────────
    print("\n── getnamed port properties ─────────────────────────────────────")
    for elem in ["ONA_1", "RING_1", "OPM_1"]:
        print(f"\n  [{elem}]")
        port_props = [
            "number of ports", "number of input ports",
            "number of output ports", "port 1", "port 2",
            "port 3", "port 4", "input", "output",
            "input 1", "input 2", "output 1",
            "ports", "pin", "port",
        ]
        for p in port_props:
            try:
                val = ic_probe.getnamed(elem, p)
                print(f"    ✓  '{p}' = {val!r}")
            except Exception:
                pass   # silent — not present

    # ─────────────────────────────────────────────────────────────────────────
    # 3. Configure ONA minimally so it has the right number of ports
    # ─────────────────────────────────────────────────────────────────────────
    print("\n── ONA minimal config ───────────────────────────────────────────")
    f_start = SPEED_OF_LIGHT / 1.560e-6
    f_stop  = SPEED_OF_LIGHT / 1.540e-6
    for cmd, lbl in [
        (f'setnamed("ONA_1","input parameter","start and stop")',   "input parameter"),
        (f'setnamed("ONA_1","number of input ports", 2)',           "n input ports = 2"),
        (f'setnamed("ONA_1","start frequency", {f_start:.6e})',     "start freq"),
        (f'setnamed("ONA_1","stop frequency",  {f_stop:.6e})',      "stop freq"),
        (f'setnamed("ONA_1","number of points", 100)',              "n points"),
    ]:
        _safe_eval(ic_probe, cmd, lbl)

    # ─────────────────────────────────────────────────────────────────────────
    # 4. Probe all connect() port name combinations
    # ─────────────────────────────────────────────────────────────────────────
    print("\n── connect() port name probe ────────────────────────────────────")

    # Candidate port names per element type
    ona_out_candidates  = ["output", "output 1", "port 1", "out"]
    ona_in_candidates   = ["input 1", "input 2", "port 1", "port 2", "in 1", "in 2"]
    ring_port_candidates = ["port 1", "port 2", "port 3", "port 4",
                            "input", "output", "through", "drop", "add"]
    opm_in_candidates   = ["input", "port 1", "in", "input 1"]

    accepted_connections = []

    def try_connect(a, pa, b, pb):
        """Try connect, then immediately disconnect to keep canvas clean."""
        cmd_con  = f'connect("{a}", "{pa}", "{b}", "{pb}");'
        cmd_dis  = f'disconnect("{a}", "{pa}");'
        try:
            ic_probe.eval(cmd_con)
            print(f"  ✓  connect('{a}','{pa}', '{b}','{pb}')")
            accepted_connections.append((a, pa, b, pb))
            try: ic_probe.eval(cmd_dis)
            except Exception: pass
            return True
        except Exception:
            print(f"  ✗  connect('{a}','{pa}', '{b}','{pb}')")
            return False

    # ── ONA output → RING_1 input ─────────────────────────────────────────
    print("\n  [ONA output → RING input]")
    found_ona_out = None
    found_ring_in = None
    for ona_out in ona_out_candidates:
        for ring_in in ring_port_candidates:
            if try_connect("ONA_1", ona_out, "RING_1", ring_in):
                found_ona_out = ona_out
                found_ring_in = ring_in
                break
        if found_ona_out:
            break

    # ── RING_1 output (through) → ONA input 1 ────────────────────────────
    print("\n  [RING through-port → ONA input 1]")
    found_ring_through = None
    found_ona_in1      = None
    for ring_out in ring_port_candidates:
        if ring_out == found_ring_in:
            continue   # skip the port already used as input
        for ona_in in ona_in_candidates:
            if try_connect("RING_1", ring_out, "ONA_1", ona_in):
                found_ring_through = ring_out
                found_ona_in1      = ona_in
                break
        if found_ring_through:
            break

    # ── RING_1 drop-port → OPM_1 input ───────────────────────────────────
    print("\n  [RING drop-port → OPM input]")
    found_ring_drop = None
    found_opm_in    = None
    for ring_drop in ring_port_candidates:
        if ring_drop in (found_ring_in, found_ring_through):
            continue
        for opm_in in opm_in_candidates:
            if try_connect("RING_1", ring_drop, "OPM_1", opm_in):
                found_ring_drop = ring_drop
                found_opm_in    = opm_in
                break
        if found_ring_drop:
            break

    # ── RING_1 remaining port → ONA input 2 ──────────────────────────────
    print("\n  [RING remaining port → ONA input 2]")
    found_ring_add = None
    found_ona_in2  = None
    used_ring_ports = {found_ring_in, found_ring_through, found_ring_drop}
    for ring_add in ring_port_candidates:
        if ring_add in used_ring_ports:
            continue
        for ona_in2 in ona_in_candidates:
            if ona_in2 == found_ona_in1:
                continue
            if try_connect("RING_1", ring_add, "ONA_1", ona_in2):
                found_ring_add = ring_add
                found_ona_in2  = ona_in2
                break
        if found_ring_add:
            break

    # ─────────────────────────────────────────────────────────────────────────
    # 5. Summary
    # ─────────────────────────────────────────────────────────────────────────
    print(f"\n{'═'*65}")
    print("  CONFIRMED PORT MAPPING")
    print(f"{'═'*65}")
    print(f"  ONA  output port     : '{found_ona_out}'")
    print(f"  ONA  input 1 port    : '{found_ona_in1}'")
    print(f"  ONA  input 2 port    : '{found_ona_in2}'")
    print(f"  RING bus-in  port    : '{found_ring_in}'   ← ONA output feeds here")
    print(f"  RING through port    : '{found_ring_through}'   ← → ONA input 1")
    print(f"  RING drop    port    : '{found_ring_drop}'   ← → OPM input")
    print(f"  RING add     port    : '{found_ring_add}'   ← → ONA input 2 (last ring)")
    print(f"  OPM  input   port    : '{found_opm_in}'")
    print(f"{'─'*65}")
    print("  WIRE FUNCTIONS for _build_circuit:")
    print(f'    wire("ONA_1",   "{found_ona_out}",   "RING_1",   "{found_ring_in}")')
    print(f'    wire("RING_1",  "{found_ring_through}",  "ONA_1",    "{found_ona_in1}")')
    print(f'    wire("RING_n",  "{found_ring_drop}",  "RING_n+1", "{found_ring_in}")')
    print(f'    wire("RING_14", "{found_ring_through}",  "ONA_1",    "{found_ona_in2}")')
    print(f'    wire("RING_n",  "{found_ring_drop}",  "OPM_k",    "{found_opm_in}")')
    print(f'    wire("RING_n",  "{found_ring_add}",   "OPM_k",    "{found_opm_in}")')
    print(f"{'═'*65}")

finally:
    try: ic_probe.close()
    except Exception: pass
    print("\nProbe session closed.")

✓ Canvas cleared

── Add elements ─────────────────────────────────────────────────
  ✓  Add ONA_1
  ✓  Add RING_1
  ✓  Add OPM_1

── getnamed port properties ─────────────────────────────────────

  [ONA_1]
    ✓  'number of input ports' = 1.0

  [RING_1]

  [OPM_1]

── ONA minimal config ───────────────────────────────────────────
  ✓  input parameter
  ✓  n input ports = 2
  ✓  start freq
  ✓  stop freq
  ✓  n points

── connect() port name probe ────────────────────────────────────

  [ONA output → RING input]
  ✓  connect('ONA_1','output', 'RING_1','port 1')

  [RING through-port → ONA input 1]
  ✓  connect('RING_1','port 2', 'ONA_1','input 1')

  [RING drop-port → OPM input]
  ✓  connect('RING_1','port 3', 'OPM_1','input')

  [RING remaining port → ONA input 2]
  ✓  connect('RING_1','port 4', 'ONA_1','input 2')

═════════════════════════════════════════════════════════════════
  CONFIRMED PORT MAPPING
═════════════════════════════════════════════════════════════════
  ONA  output

In [18]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  DIAGNOSTIC — find ALL ports of UNIDIRECTIONAL ring                        ║
# ║  Strategy: rebuild canvas from scratch for every candidate port test       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

SPEED_OF_LIGHT = 299_792_458.0

def _fresh_canvas(ic):
    """Guaranteed clean state — rebuild ONA + UNIDIR RING + OPM every call."""
    ic.eval("switchtodesign; selectall; delete;")
    # ONA with 2 input ports
    ic.eval('addelement("Optical Network Analyzer"); set("name","ONA_1");')
    ic.eval('setnamed("ONA_1","input parameter","start and stop");')
    ic.eval('setnamed("ONA_1","number of input ports", 2);')
    f_s = SPEED_OF_LIGHT / 1.560e-6
    f_e = SPEED_OF_LIGHT / 1.540e-6
    ic.eval(f'setnamed("ONA_1","start frequency",{f_s:.6e});')
    ic.eval(f'setnamed("ONA_1","stop frequency", {f_e:.6e});')
    ic.eval('setnamed("ONA_1","number of points", 100);')
    # UNIDIRECTIONAL ring
    ic.eval('addelement("Double Bus Ring Resonator"); set("name","RING_1");')
    ic.eval('setnamed("RING_1","configuration","unidirectional");')
    # Second ring for cascade test
    ic.eval('addelement("Double Bus Ring Resonator"); set("name","RING_2");')
    ic.eval('setnamed("RING_2","configuration","unidirectional");')
    # OPM
    ic.eval('addelement("Optical Power Meter"); set("name","OPM_1");')

def _try_connect(ic, a, pa, b, pb):
    try:
        ic.eval(f'connect("{a}","{pa}","{b}","{pb}");')
        return True
    except Exception:
        return False

ic_probe = lumapi.INTERCONNECT(hide=False)

try:
    # ── Expand candidate list significantly ──────────────────────────────────
    ring_candidates = [
        "input", "output", "through", "drop", "add",
        "port 1", "port 2", "port 3", "port 4",
        "input 1", "input 2", "output 1", "output 2",
        "pass", "pass output", "through output",
        "drop output", "add input", "add output",
        "bus 1 input", "bus 1 output", "bus 2 input", "bus 2 output",
        "in", "out", "in 1", "in 2", "out 1", "out 2",
    ]
    ona_out   = "output"
    ona_in    = ["input 1", "input 2"]
    opm_in    = ["input", "port 1", "in"]

    # ── STEP 1: find RING input port (ONA → RING) ────────────────────────────
    print("── STEP 1: ONA output → RING ? ──────────────────────────────────")
    ring_input_port = None
    for rp in ring_candidates:
        _fresh_canvas(ic_probe)
        if _try_connect(ic_probe, "ONA_1", ona_out, "RING_1", rp):
            print(f"  ✓  ONA 'output' → RING '{rp}'")
            ring_input_port = rp
            break
        else:
            print(f"  ✗  RING '{rp}'")

    print(f"\n  → RING input port = '{ring_input_port}'\n")

    # ── STEP 2: find RING through/output port (RING → ONA input 1) ───────────
    print("── STEP 2: RING ? → ONA 'input 1' ──────────────────────────────")
    ring_through_port = None
    for rp in ring_candidates:
        if rp == ring_input_port:
            continue
        _fresh_canvas(ic_probe)
        # Pre-wire the known input so the ring is "active"
        _try_connect(ic_probe, "ONA_1", ona_out, "RING_1", ring_input_port)
        if _try_connect(ic_probe, "RING_1", rp, "ONA_1", "input 1"):
            print(f"  ✓  RING '{rp}' → ONA 'input 1'")
            ring_through_port = rp
            break
        else:
            print(f"  ✗  RING '{rp}'")

    print(f"\n  → RING through port = '{ring_through_port}'\n")

    # ── STEP 3: find RING drop port (RING → RING_2 input) ────────────────────
    print("── STEP 3: RING_1 ? → RING_2 (cascade drop) ────────────────────")
    ring_drop_port = None
    used = {ring_input_port, ring_through_port}
    for rp in ring_candidates:
        if rp in used:
            continue
        _fresh_canvas(ic_probe)
        _try_connect(ic_probe, "ONA_1",  ona_out,          "RING_1", ring_input_port)
        _try_connect(ic_probe, "RING_1", ring_through_port, "ONA_1", "input 1")
        if _try_connect(ic_probe, "RING_1", rp, "RING_2", ring_input_port):
            print(f"  ✓  RING_1 '{rp}' → RING_2 '{ring_input_port}'")
            ring_drop_port = rp
            break
        else:
            print(f"  ✗  RING_1 '{rp}'")

    print(f"\n  → RING drop port = '{ring_drop_port}'\n")

    # ── STEP 4: find RING add port (RING → OPM) ───────────────────────────────
    print("── STEP 4: RING_1 ? → OPM ───────────────────────────────────────")
    ring_add_port = None
    opm_input_port = None
    used = {ring_input_port, ring_through_port, ring_drop_port}
    for rp in ring_candidates:
        if rp in used:
            continue
        for op in opm_in:
            _fresh_canvas(ic_probe)
            _try_connect(ic_probe, "ONA_1",  ona_out,           "RING_1", ring_input_port)
            _try_connect(ic_probe, "RING_1", ring_through_port,  "ONA_1", "input 1")
            if ring_drop_port:
                _try_connect(ic_probe, "RING_1", ring_drop_port, "RING_2", ring_input_port)
            if _try_connect(ic_probe, "RING_1", rp, "OPM_1", op):
                print(f"  ✓  RING_1 '{rp}' → OPM '{op}'")
                ring_add_port  = rp
                opm_input_port = op
                break
        if ring_add_port:
            break
        print(f"  ✗  RING '{rp}' → OPM (all candidates)")

    print(f"\n  → RING add port = '{ring_add_port}'")
    print(f"  → OPM input port = '{opm_input_port}'\n")

    # ── STEP 5: verify full cascade wiring on clean canvas ───────────────────
    print("── STEP 5: full cascade verify (ONA→R1→R2→OPM) ─────────────────")
    _fresh_canvas(ic_probe)
    results = [
        _try_connect(ic_probe, "ONA_1",  ona_out,            "RING_1", ring_input_port),
        _try_connect(ic_probe, "RING_1", ring_through_port,   "ONA_1", "input 1"),
        _try_connect(ic_probe, "RING_1", ring_drop_port,      "RING_2", ring_input_port),
        _try_connect(ic_probe, "RING_2", ring_through_port,   "ONA_1", "input 2"),
        _try_connect(ic_probe, "RING_2", ring_add_port,       "OPM_1", opm_input_port),
    ]
    labels = [
        f"ONA 'output'        → RING_1 '{ring_input_port}'",
        f"RING_1 '{ring_through_port}' → ONA 'input 1'",
        f"RING_1 '{ring_drop_port}'    → RING_2 '{ring_input_port}'",
        f"RING_2 '{ring_through_port}' → ONA 'input 2'",
        f"RING_2 '{ring_add_port}'     → OPM '{opm_input_port}'",
    ]
    for ok, lbl in zip(results, labels):
        print(f"  {'✓' if ok else '✗'}  {lbl}")

    # ── Final summary ─────────────────────────────────────────────────────────
    print(f"\n{'═'*65}")
    print("  UNIDIRECTIONAL RING — FINAL CONFIRMED PORT MAP")
    print(f"{'═'*65}")
    print(f"  RING input   (← ONA output / ← prev drop) : '{ring_input_port}'")
    print(f"  RING through (→ ONA input 1/2)             : '{ring_through_port}'")
    print(f"  RING drop    (→ next RING input)            : '{ring_drop_port}'")
    print(f"  RING add     (→ OPM input)                  : '{ring_add_port}'")
    print(f"  OPM  input                                  : '{opm_input_port}'")
    print(f"{'─'*65}")
    print("  wire() calls for _build_circuit:")
    print(f'    wire(ONA,    "{ona_out}",          RING_1,   "{ring_input_port}")')
    print(f'    wire(RING_1, "{ring_through_port}", ONA,      "input 1")')
    print(f'    wire(RING_n, "{ring_drop_port}",    RING_n+1, "{ring_input_port}")')
    print(f'    wire(RING_14,"{ring_through_port}", ONA,      "input 2")')
    print(f'    wire(RING_n, "{ring_add_port}",     OPM_k,    "{opm_input_port}")')
    print(f"{'═'*65}")

finally:
    try: ic_probe.close()
    except Exception: pass
    print("\nProbe closed.")

── STEP 1: ONA output → RING ? ──────────────────────────────────
  ✓  ONA 'output' → RING 'input'

  → RING input port = 'input'

── STEP 2: RING ? → ONA 'input 1' ──────────────────────────────
  ✗  RING 'output'
  ✗  RING 'through'
  ✗  RING 'drop'
  ✗  RING 'add'
  ✗  RING 'port 1'
  ✗  RING 'port 2'
  ✗  RING 'port 3'
  ✗  RING 'port 4'
  ✗  RING 'input 1'
  ✗  RING 'input 2'
  ✓  RING 'output 1' → ONA 'input 1'

  → RING through port = 'output 1'

── STEP 3: RING_1 ? → RING_2 (cascade drop) ────────────────────
  ✗  RING_1 'output'
  ✗  RING_1 'through'
  ✗  RING_1 'drop'
  ✗  RING_1 'add'
  ✗  RING_1 'port 1'
  ✗  RING_1 'port 2'
  ✗  RING_1 'port 3'
  ✗  RING_1 'port 4'
  ✗  RING_1 'input 1'
  ✗  RING_1 'input 2'
  ✓  RING_1 'output 2' → RING_2 'input'

  → RING drop port = 'output 2'

── STEP 4: RING_1 ? → OPM ───────────────────────────────────────
  ✗  RING 'output' → OPM (all candidates)
  ✗  RING 'through' → OPM (all candidates)
  ✗  RING 'drop' → OPM (all candidates)
  ✗ 